# FLUX.1 [schnell] with 🤗 Diffusers — free-tier Colab

FLUX.1 [schnell] is the 4-step, Apache-2.0 FLUX model from Black Forest Labs. The full model (12B transformer + T5-XXL text encoder) does not fit a free T4 or free-tier RAM in bf16, so this notebook loads the transformer and the T5 encoder in **4-bit NF4** (bitsandbytes) and offloads idle components to the CPU.

The model repo asks you to accept its licence once: open https://huggingface.co/black-forest-labs/FLUX.1-schnell while logged in, accept, then paste a read token below.

Part of the SoapBox GenAI school (hathor.soapbox.community/colab). Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
!pip -q install -U diffusers transformers accelerate bitsandbytes sentencepiece protobuf

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
from diffusers import DiffusionPipeline
from diffusers.quantizers import PipelineQuantizationConfig

quant = PipelineQuantizationConfig(
    quant_backend="bitsandbytes_4bit",
    quant_kwargs={"load_in_4bit": True, "bnb_4bit_quant_type": "nf4", "bnb_4bit_compute_dtype": torch.bfloat16},
    components_to_quantize=["transformer", "text_encoder_2"],
)
pipe = DiffusionPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    quantization_config=quant,
    torch_dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()

In [ ]:
prompt = "a temple of Hathor at dawn, carved sandstone columns, gold light, highly detailed"

image = pipe(
    prompt,
    guidance_scale=0.0,          # schnell is guidance-distilled: no CFG
    num_inference_steps=4,       # schnell is trained for 1-4 steps
    max_sequence_length=256,
    height=768, width=768,
    generator=torch.Generator("cpu").manual_seed(0),
).images[0]
image.save("flux-schnell.png")
image